In [7]:
import os
import shutil
import cv2
import numpy as np
import random
from tqdm import tqdm

def manual_augment(img):
    """Applies your specific CNN parameters using only OpenCV/NumPy."""
    h, w = img.shape[:2]

    # 1. Rotation (15 degrees)
    angle = random.uniform(-15, 15)
    M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1)
    img = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REPLICATE)

    # 2. Width/Height Shift (10%)
    tx = random.uniform(-0.1, 0.1) * w
    ty = random.uniform(-0.1, 0.1) * h
    T = np.float32([[1, 0, tx], [0, 1, ty]])
    img = cv2.warpAffine(img, T, (w, h), borderMode=cv2.BORDER_REPLICATE)

    # 3. Zoom (15%)
    zoom = random.uniform(0.85, 1.15)
    new_h, new_w = int(h * zoom), int(w * zoom)
    img_zoomed = cv2.resize(img, (new_w, new_h))
    if zoom > 1: # Crop
        start_h = (new_h - h) // 2
        start_w = (new_w - w) // 2
        img = img_zoomed[start_h:start_h+h, start_w:start_w+w]
    else: # Pad
        pad_h = (h - new_h) // 2
        pad_w = (w - new_w) // 2
        img = cv2.copyMakeBorder(img_zoomed, pad_h, h-new_h-pad_h, pad_w, w-new_w-pad_w, cv2.BORDER_REPLICATE)

    # 4. Brightness (0.8 to 1.2)
    brightness = random.uniform(0.8, 1.2)
    img = cv2.convertScaleAbs(img, alpha=brightness, beta=0)

    # 5. Horizontal Flip
    if random.choice([True, False]):
        img = cv2.flip(img, 1)

    return img

def create_cnn_dataset_no_pil(source_root, target_root):
    classes = ['Covid-19', 'Normal', 'Pneumonia-Bacterial', 'Pneumonia-Viral']

    for cls in classes:
        source_dir = os.path.join(source_root, cls)
        target_dir = os.path.join(target_root, cls)
        os.makedirs(target_dir, exist_ok=True)

        images = [f for f in os.listdir(source_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        print(f"\n🚀 Processing {cls}...")

        for img_name in tqdm(images):
            img_path = os.path.join(source_dir, img_name)
            img = cv2.imread(img_path)
            if img is None: continue

            # Standardize size for CNN (DenseNet/ViT input)
            img = cv2.resize(img, (224, 224))

            # Save Original
            cv2.imwrite(os.path.join(target_dir, f"orig_{img_name}"), img)

            # Create exactly one unique Augmented version
            aug_img = manual_augment(img)
            cv2.imwrite(os.path.join(target_dir, f"aug_{img_name}"), aug_img)

    print(f"\n✨ Done! Balanced dataset (approx. 5,392 per class) ready in: {target_root}")

# --- PATHS ---
SOURCE = "/Users/kpvarma/PycharmProjects/CNN_ViT_Hybrid_Vit_Research_Pneumonia_4_Classification/Equal_sample"
DESTINATION = "/Users/kpvarma/PycharmProjects/CNN_ViT_Hybrid_Vit_Research_Pneumonia_4_Classification/CNN_Augmented_Data"

create_cnn_dataset_no_pil(SOURCE, DESTINATION)


🚀 Processing Covid-19...


100%|██████████| 2696/2696 [00:03<00:00, 765.18it/s]



🚀 Processing Normal...


100%|██████████| 2696/2696 [00:03<00:00, 763.49it/s]



🚀 Processing Pneumonia-Bacterial...


100%|██████████| 2696/2696 [00:03<00:00, 793.86it/s]



🚀 Processing Pneumonia-Viral...


100%|██████████| 2696/2696 [00:03<00:00, 739.48it/s]


✨ Done! Balanced dataset (approx. 5,392 per class) ready in: /Users/kpvarma/PycharmProjects/CNN_ViT_Hybrid_Vit_Research_Pneumonia_4_Classification/CNN_Augmented_Data
